# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql.functions import col, concat_ws, sum as spark_sum
import tempfile
import os

In [0]:
# Extract
bookings = spark.sql("SELECT * FROM bookings")
members = spark.sql("SELECT * FROM members")
facilities = spark.sql("SELECT * FROM facilities")

In [0]:
# Transform
bookings_df = (bookings
               .where(
                   (col("starttime") >= "2012-09-01") &
                   (col("starttime") < "2012-10-01")
                )
               .groupBy("facid")
               .agg(
                   spark_sum(col("slots")).alias("total_slots")
                )
               .orderBy("total_slots")
                )

display(bookings_df)

facid,total_slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


In [0]:
# Load
bookings_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("/FileStore/tables/bookings_df_parquet")

## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Transformation
result = (
    bookings
    .join(facilities, "facid")
    .join(members, "memid")
    .where(col("name").like("Tennis Court%"))
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("name").alias("facility")
    )
    .distinct()
    .orderBy("member")
)

display(result)

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 2


In [0]:
# Load
result.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("facility") \
    .saveAsTable("default.threejoin_delta")

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import time
import requests
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col, to_date, weekofyear, year, max as spark_max

In [0]:
lst = ["GOOG", "AAPL", "MSFT", "TSLA"]
rows = []

url = "https://alpha-vantage.p.rapidapi.com/query"

headers = {
            "x-rapidapi-key": "2b331f1bd2msh6b23ae18d71fbf2p17c452jsn2f3f6ae3d60c",
            "x-rapidapi-host": "alpha-vantage.p.rapidapi.com",
            "Content-Type": "application/json"
        }

for symbol in lst:
    querystring = {"function":"TIME_SERIES_DAILY",
                "symbol": symbol,
                "outputsize":"compact",
                "datatype":"json"}

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    if 'Time Series (Daily)' not in data:
        print(f"Warning: No data for {symbol}. Response: {data}")
        time.sleep(15)
        continue

    daily_data = data['Time Series (Daily)']

    for date_str, values in daily_data.items():
        close_price = float(values["4. close"])

        rows.append(Row(
            symbol = symbol,
            date = date_str,
            close = close_price
        ))

    time.sleep(15)


# Create Spark DataFrame
daily_df = spark.createDataFrame(rows)

weekly_max_df = (
    daily_df
    .withColumn("date", to_date(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("week", weekofyear(col("date")))
    .groupBy("symbol", "year", "week")
    .agg(
        spark_max("close").alias("weekly_max_close")
    )
    .orderBy("symbol", "year", "week")
)

display(weekly_max_df)

symbol,year,week,weekly_max_close
AAPL,2025,1,273.08
AAPL,2026,1,271.01
AAPL,2026,2,267.26
AAPL,2026,3,261.05
AAPL,2026,4,248.35
AAPL,2026,5,259.48
AAPL,2026,6,278.12
AAPL,2026,7,275.5
AAPL,2026,8,264.58
AAPL,2026,9,274.23


In [0]:
# Load
weekly_max_df.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("symbol") \
    .saveAsTable("default.weekly_max_df")

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Solution
rna_table = (spark.read
  .format("jdbc")
  .option("url", "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs")
  .option("user", "reader")
  .option("driver", "org.postgresql.Driver")
  .option("password", "NWDMCE5xdipIjRrp")
  .load()
)

rna_table.printSchema

In [0]:
# Extract
result = rna_table.select("*").limit(100)
display(result)

id upi timestamp userstamp crc64 len seq_short seq_long md5 39409471 URS000259573F 2022-09-05T20:15:34.477153Z rnacen 97EDC0E905DEBD6C 1318 AATCTATTTATTTTTTAAAAAATTAATTATTAAAAATAATTATATAAATTATTGTTTAAGTATATTATATAGAATAAATATTATTAATTATAGTTAATTAGTATTGTGAAAGAATATTGAAATAAAATGAAAAATATTTATTTTAAAAGAAAATTTAATTTATTGTATCTTGTGTATCAGAGTATATTAAATAAAAAATAATTATTTATTTTTCTCGATTTTAAAAGATTTAATAAATTATAAAAGTTAATGTAATAAAATTATTTTTAATAATTTATTAGAAATGAAATGTTATTCGTTTTTAAAGGTATCTAGTTTTTTAAGAAATAAATTTAATTTAGTTTTTTAAAATTTATTTATTTGTTTAGTTAATTAAATAAATTTTAAAAAATAATATTTTATGGGATAAGCTATAAAATAAATTTTTAAAAATAATAAATAATGTTTAAATTTTATATGTTTAAGATTATCAATTATTATTAAGTTTGTTATAAATTAATTTTTTTTAATAAATTATTTATTATATTTTAAGTTTTTATTAAAATATTTATTTTAATATTAAAAATAAAGAAATAATGATATAATTAGTATATTTTAATGTATTTATATGTATATAAATGATAAGTTTGTGTAAAGAATTCGGCAAAAATAATGTTCGCCTGTTTAACAAAAACATGTCTTTTAGAATTTAATTTAAAGTCTGACCTGCCCACTGAATTTTTTTAAATGGCCGCAGTATTTTAACTGTGCAAAGGTAGCATAATCATTAGTCTTTTAATTGAAGGCTGGAATGAATGGTTGGACGAAATATTAACTGTTTCATTTAAATTTATAATAAAATTTTATTTTTTAGTCAAAAAGCTAAAATAAATTTAAAAGACGAGAAGACCCTATAAATCTTTATATATAGTTTATTTTAATTTTATAGATAGATTTAATTATGATAATATTTATATATTTTATTGGGGTGATATTAAAATTTAATAAACTTTTAAAATTTAAATCATTAATTTATGAATAATTGATCCATTTTTAATGATTAAAAAATTAAGTTACTTTAGGGATAACAGCGTAATTTTTTTGGAGAGTTCATATTGATAAAAAAGATTGCGACCTCGATGTTGGATTAAGATATAATTTTAGGTGCAGCCGTTTAAATTTAAAGTCTGTTCGACTTTTAAAATCTTACATGATCTGAGTTCAAACCGGTGTAAGCCAGGTTGGTTTCTATCTTTAATAAATTAATATATTTTAGTACGAAAGGATTAAATATATAAATAATTGTATTTTATAAATAGAATATTATTAATAATTTATA null a4bf94890e6eb5ef1fb072993a5eebb9 39409472 URS0002595740 2022-09-05T20:15:34.477214Z rnacen 43A6B3D53F65E149 1556 GGAAGGATCATTGAATCTATCACAATCCACAACCGTGAAATATTTTGTTGGCCTCCATCCGTGCACTTGTGCGGGGATGGGCCGGCTTTGCTCTGATTTAATTCGGGGTGAAGGCGGGGCCTGCGCTTGTCTACTTCCTTCGGGAACGGGCAGGTGTGGGTCGGTCTTTATTAACCAACCAACACCAAACCAAAATTCTAAATATAACGAGTGCGTGGCTTAGAGCCAACACTCACTAACCAAGACAACTCTCAACAACGGATATCTTGGCTCTCGTAACGATGAAGAACGCAGCGAAATGCGATACGTAATGTGAATTGCAGAAATACGTGAACCATCGAATCTTTGAACGCATATTGCGCTCGCGGCCTCGGCCAAGAGCATGTCTGCCTCAGCGTCGGGTTAAAACTCGCCCTACTCAATTCATTGGGTATGGTGCGGATCTGGCTTTCCCGGTTATTCAATTAACCGGGTTAGCTGAAGAGCAGAGGTTGATGCATGGACCCGCTAAGGGCCTCGACTGGGTAGGCAATTCGTTGCTGATGCTTTAGTCGGTGGTCTGGATCTGTGCTTGTCGACCCAAACCAGGAACTTGGCTTCTGCCAAGAAAACCCCTTTATCCTCGACCTGAGCTCAGGCAAGAACACCCGCTGAACTTAAGCATATCAATAAGCGGAGGAAAAGAAACTAACAAGGATTCCCCTAGTAACGGCGAGCGAACCGGGAATAGCCCAACTTGAAAATCTCCCTTTGGAGAATTGTAGTCTATCGAAGCGCCCTCAGCAGGAGGCAGAGCTCAAGTCGGATCGAATGCCGCGTCAGAGAGGGTGATAACCCCGTCGGCTCTTGCTCTTATCTGCAACACGAGGTGCTTTCCACGAGTCGGGTTGTTTGGGAATGCAGCCCTAATTTGGAGGTAAATCCCTTCTAAGGCTAAATACTGACGAGAGACCGATAGCGAACAAGTACCGTGAGGGAAAGATGAAAAGAACTTTGAAAAGAGAGTTAAAAGTGCTTGAAATTGTTGAGGGGGAAGCGTTTGGAGTTCGTAGGTGCGCCCAGGCTTAAGCAATCCTAACGGATTGTTGAATGTGCTGGGTGCTGGTCAATGTGGATTGGCTTGGCGGGATAACAGTTGGGTCTACCCAGGTAACCTAACCAATGCCGCCGAGCCGATCAAGGTGTGAAGGGTGCTCTGTCCTCCGGGATCTGCATCCTAAAGACATTGGCAGAAGAGCTTCAACCGGCCCGTCTTGAAACACGGACCAAGGAGTCTAACATGTATGCGAGTTGGCGGGTGGTAAACCCGTAAGCGCAAGTAACCTGACTGGTGGGAGGGCCTGTGCCTGCACCATCGACCGACCATGTTGCTTTTGCGAAAGGTTTGAGTGCGAGCATACCTGTTGGGACCCGAAAGATGGTGAACTATGCCTGAGCAGGGTGAAGCCAGAGGAAACTCTGGTGGAGGCTCGTAGATGTGCTGACGTGCAAATCGCTTTTCAGACTTGGGTATAGGGGCGAAAGACTAATCGAACCATCTAGTAGCTGGTTCCCTC null a4bf95388089ebcb22d755c2000170ba 39409473 URS0002595741 2022-09-05T20:15:34.477258Z rnacen 92693BDBCCE0D10F 4767 null CAGCGCAGTAGTCCTTGCCTTGTCTTAGTGCGGTGTGCACGGCGGGGCGGCGCGGAGGTGGCGTTTAGGGCTCGGCGATCCTTAGGGGGTTGGGCTCACCGGGCCGCGGCGCGGAGGAGAGACCGACCGAGAGAGAGGGGGCAGGCAGGCAGGCAGGCGCGTTTGTTTGGTGTAAGGGAAGGGAGGGAGGTGGTGTGCATCGCGCGGCGTGGGCGTGCCCGCCGGGCGGAGGCGGAGGTGTGCTGTGGTCGGCTAGGGGAGGGGAAGGTGTGGACTGCGGCTGCTCAGGGCGCACCCGCCGCCGCGGCCGCTACGGGGGGCGCGACTGCCGCACTATGGGTCTCGCTGGGTCGCTTATAGGCTTAGGGGGTCGGTTGTTGTGGTGGTGCCGGAGGGGGAGGGAGGGAGGGAGTGAAGGGGTGGCGCTTAGGCCCCCTGCGGAGGAGGAGCGGAGAGAGAGGGAGAGGGAGAGGGAGAGCGGAGGGAAGGGGGCGGGGATTTCGCGCTGGCGTGGCGGCGGGCCTCGGGTCGGCGGGTTCAGCCGACGAGCGGGGGAGAGGGCGAACGAGTCGGCTGGGGCCCTCGGGTCCTAGCACGGTCTCCGTCCGTCCGTCCTTTCCCCGATTGCTCTCGTCGACCAACCCTGACCACCTAACCGTTAGCACACTACATGAACCGAGACGGCGACTAGGTTGAGTTCGGTCGAACTAGTGCGTCGGAGCCGTGTGGTACCGCGGTAGGCG

In [0]:
# Load
result.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("default.rna_100_records")